# 📊 Experimental Validation

This notebook demonstrates that `synth-xtal` produces physically accurate mathematical outputs by comparing its simulated $F_{calc}$ structure factors to peer-reviewed experimental $F_{obs}$ deposited in the PDB.

In [ ]:
!pip install synth-xtal reciprocalspaceship scipy matplotlib

## 1. Fetch Experimental Data
We use PDB 4LZT, a 0.95A Lysozyme structure published in Acta Cryst. D.

In [ ]:
import urllib.request
import gemmi

# Download atomic coordinates and structure factors
urllib.request.urlretrieve("https://files.rcsb.org/download/4LZT.pdb", "4LZT.pdb")
urllib.request.urlretrieve("https://files.rcsb.org/download/4LZT-sf.cif", "4LZT-sf.cif")

# Strip ANISOU to prevent NaN issues in basic simulation
st = gemmi.read_structure("4LZT.pdb")
for m in st:
    for c in m:
        for r in c:
            for a in r:
                a.aniso = gemmi.SMat33f(0, 0, 0, 0, 0, 0)
st.write_pdb("4LZT_clean.pdb")

## 2. Generate Synthetic Data

In [ ]:
from synth_xtal.simulator import simulate_diffraction

simulate_diffraction("4LZT_clean.pdb", "simulated.mtz", d_min=1.5)

## 3. Correlate with Physics

In [ ]:
import reciprocalspaceship as rs
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

exp_ds = rs.read_cif("4LZT-sf.cif")
calc_ds = rs.read_mtz("simulated.mtz")

# Filter negative intensities and convert to F_obs
exp_ds = exp_ds[exp_ds["IMEAN"] > 0]
exp_ds["F_obs"] = np.sqrt(exp_ds["IMEAN"])
calc_ds.rename(columns={"FC": "F_calc"}, inplace=True)

# Merge datasets by matching Miller Indices (H, K, L)
merged = exp_ds.merge(calc_ds, left_index=True, right_index=True, how="inner")

# Convert to numpy and drop NaNs
f_calc = merged["F_calc"].astype(float).to_numpy()
f_obs = merged["F_obs"].astype(float).to_numpy()
mask = ~np.isnan(f_calc) & ~np.isnan(f_obs)
f_calc_clean, f_obs_clean = f_calc[mask], f_obs[mask]

corr, _ = pearsonr(f_calc_clean, f_obs_clean)
print(f"Pearson Correlation vs Experimental F_obs: {corr:.3f}")

plt.figure(figsize=(6, 6))
plt.scatter(f_obs_clean, f_calc_clean, alpha=0.1, s=2)
plt.xlabel("Experimental F_obs")
plt.ylabel("Synthetic F_calc")
plt.title("synth-xtal vs 4LZT Evidence")
plt.show()